# Causal Uplift Modeling for Email Marketing Optimization

A complete reproducible analysis of the uploaded Hillstrom email experiment.

## 1. Project Objective

Estimate average and heterogeneous treatment effects and convert them into a practical policy for Men's Email, Women's Email, or No Email.

In [1]:

from pathlib import Path
import sys, pandas as pd, numpy as np
ROOT=Path.cwd().resolve()
if not (ROOT/"src").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from src.data_utils import load_data, prepare, validate_data
from src.experiment_analysis import all_ates
from src.config import DATA_PATH, RANDOM_SEED, TEST_SIZE


## 2. Dataset Loading

In [2]:

df_raw=load_data(DATA_PATH)
print("Shape:", df_raw.shape)
display(df_raw.head())
print(df_raw.dtypes)


Shape: (64000, 12)


,recency,history_segment,history,mens,womens,zip_code,newbie,channel,segment,visit,conversion,spend
0,10,2) $100 - $200,142.44,1,0,Surburban,0,Phone,Womens E-Mail,0,0,0.0
1,6,3) $200 - $350,329.08,1,1,Rural,1,Web,No E-Mail,0,0,0.0
2,7,2) $100 - $200,180.65,0,1,Surburban,1,Web,Womens E-Mail,0,0,0.0
3,9,5) $500 - $750,675.83,1,0,Rural,1,Web,Mens E-Mail,0,0,0.0
4,2,1) $0 - $100,45.34,1,0,Urban,0,Web,Womens E-Mail,0,0,0.0


recency              int64
history_segment     object
history            float64
mens                 int64
womens               int64
zip_code            object
newbie               int64
channel             object
segment             object
visit                int64
conversion           int64
spend              float64
dtype: object


## 3. Data Quality

In [3]:

report=validate_data(df_raw)
display(pd.DataFrame({"missing":pd.Series(report["missing"])}))
print("Duplicates:",report["duplicates"])
print("Treatment counts:",report["treatment_counts"])


,missing
recency,0
history_segment,0
history,0
mens,0
womens,0
zip_code,0
newbie,0
channel,0
segment,0
visit,0


Duplicates: 6562
Treatment counts: {'Womens E-Mail': 21387, 'Mens E-Mail': 21307, 'No E-Mail': 21306}


In [4]:

df=prepare(df_raw)
display(df.describe(include="all").T)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
recency,64000.0,NaN,NaN,NaN,5.763734,3.507592,1.0,2.0,6.0,9.0,12.0
history_segment,64000,7,1) $0 - $100,22970,NaN,NaN,NaN,NaN,NaN,NaN,NaN
history,64000.0,NaN,NaN,NaN,242.085656,256.158608,29.99,64.66,158.11,325.6575,3345.93
mens,64000.0,NaN,NaN,NaN,0.551031,0.497393,0.0,0.0,1.0,1.0,1.0
womens,64000.0,NaN,NaN,NaN,0.549719,0.497526,0.0,0.0,1.0,1.0,1.0
zip_code,64000,3,Surburban,28776,NaN,NaN,NaN,NaN,NaN,NaN,NaN
newbie,64000.0,NaN,NaN,NaN,0.50225,0.499999,0.0,0.0,1.0,1.0,1.0
channel,64000,3,Web,28217,NaN,NaN,NaN,NaN,NaN,NaN,NaN
segment,64000,3,Womens E-Mail,21387,NaN,NaN,NaN,NaN,NaN,NaN,NaN
visit,64000.0,NaN,NaN,NaN,0.146781,0.35389,0.0,0.0,0.0,0.0,1.0


## 4. Exploratory Data Analysis

In [5]:

display(df.groupby("segment")[["visit","conversion","spend"]].mean().round(4))


,visit,conversion,spend
segment,,,
Mens E-Mail,0.1828,0.0125,1.4226
No E-Mail,0.1062,0.0057,0.6528
Womens E-Mail,0.1514,0.0088,1.0772


## 5. Treatment Assignment and Randomization Validation

In [6]:

display(df["segment"].value_counts())
balance=df.groupby("segment")[["recency","history","mens","womens","newbie"]].mean().T
display(balance.round(3))


segment
Womens E-Mail    21387
Mens E-Mail      21307
No E-Mail        21306
Name: count, dtype: Int64

segment,Mens E-Mail,No E-Mail,Womens E-Mail
recency,5.774,5.750,5.768
history,242.836,240.883,242.537
mens,0.551,0.553,0.549
womens,0.551,0.548,0.550
newbie,0.502,0.502,0.503


The experiment has three arms: Men's Email, Women's Email, and No Email. Balance is inspected rather than assumed to be perfect.

## 6. ATE Analysis

In [7]:

ates=all_ates(df)
display(ates[ates.outcome=="conversion"].round(6))


,treatment,outcome,treatment_mean,control_mean,absolute_lift,relative_lift,ci_low,ci_high,p_value
3,Mens E-Mail,conversion,0.012531,0.005726,0.006805,1.188422,0.005000,0.008610,0.000000
4,Womens E-Mail,conversion,0.008837,0.005726,0.003111,0.543313,0.001499,0.004724,0.000156
5,Any Email,conversion,0.010681,0.005726,0.004955,0.865263,0.003548,0.006361,0.000000


## 7. Visit Effects

In [8]:
display(ates[ates.outcome=="visit"].round(6))

,treatment,outcome,treatment_mean,control_mean,absolute_lift,relative_lift,ci_low,ci_high,p_value
0,Mens E-Mail,visit,0.182757,0.106167,0.076590,0.721405,0.069953,0.083226,0.0
1,Womens E-Mail,visit,0.151400,0.106167,0.045233,0.426055,0.038894,0.051573,0.0
2,Any Email,visit,0.167049,0.106167,0.060882,0.573453,0.055438,0.066325,0.0


## 8. Conversion Effects

In [9]:
display(ates[ates.outcome=="conversion"].round(6))

,treatment,outcome,treatment_mean,control_mean,absolute_lift,relative_lift,ci_low,ci_high,p_value
3,Mens E-Mail,conversion,0.012531,0.005726,0.006805,1.188422,0.005000,0.008610,0.000000
4,Womens E-Mail,conversion,0.008837,0.005726,0.003111,0.543313,0.001499,0.004724,0.000156
5,Any Email,conversion,0.010681,0.005726,0.004955,0.865263,0.003548,0.006361,0.000000


## 9. Spend Effects

In [10]:
display(ates[ates.outcome=="spend"].round(6))

,treatment,outcome,treatment_mean,control_mean,absolute_lift,relative_lift,ci_low,ci_high,p_value
6,Mens E-Mail,spend,1.422617,0.652789,0.769827,1.179289,0.485144,1.054510,0.000000
7,Womens E-Mail,spend,1.077202,0.652789,0.424412,0.650152,0.168960,0.679865,0.001128
8,Any Email,spend,1.249585,0.652789,0.596796,0.914225,0.376186,0.817406,0.000000


## 10. Feature Engineering

In [11]:

from src.feature_engineering import make_preprocessor
pre=make_preprocessor()
X=pre.fit_transform(df)
print("Encoded feature matrix:",X.shape)


Encoded feature matrix: (64000, 18)


Outcome variables are excluded from the feature set. Treatment assignment is also excluded from the customer predictors used by the separate treatment/control models.

## 11. Train/Test Split

In [12]:

from sklearn.model_selection import train_test_split
idx_train,idx_test=train_test_split(np.arange(len(df)),test_size=TEST_SIZE,random_state=RANDOM_SEED,stratify=df.segment)
train=df.iloc[idx_train].reset_index(drop=True)
test=df.iloc[idx_test].reset_index(drop=True)
print(train.shape,test.shape)
display(pd.concat([train.segment.value_counts(normalize=True),test.segment.value_counts(normalize=True)],axis=1,keys=["train","test"]))


(48000, 12) (16000, 12)


,train,test
segment,,
Womens E-Mail,0.334167,0.334188
Mens E-Mail,0.332917,0.332937
No E-Mail,0.332917,0.332875


## 12. Two-Model Uplift

In [13]:

from src.uplift_models import train_tlearner, predict_tlearner
Xtr=pre.fit_transform(train); Xte=pre.transform(test)
for arm in ["Mens E-Mail","Womens E-Mail"]:
    mt,mc=train_tlearner(Xtr,train.segment.values,train.conversion.values,arm)
    pt,pc=predict_tlearner(mt,mc,Xte)
    print(arm,"mean predicted treatment:",pt.mean(),"control:",pc.mean(),"uplift:",(pt-pc).mean())


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Mens E-Mail mean predicted treatment: 0.01249606732630798 control: 0.004640575517347787 uplift: 0.007855491808960191


Womens E-Mail mean predicted treatment: 0.008325929030301804 control: 0.004640575517347787 uplift: 0.0036853535129540162


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 13. Class Transformation

In [14]:

from src.uplift_models import train_class_transformation
for arm in ["Mens E-Mail","Womens E-Mail"]:
    cm,p=train_class_transformation(Xtr,train.segment.values,train.conversion.values,arm)
    print(arm,"empirical propensity:",p,"mean estimated transformed effect:",cm.predict(Xte).mean())


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Mens E-Mail empirical propensity: 0.5 mean estimated transformed effect: 0.008050927121215008


Womens E-Mail empirical propensity: 0.5009369144284822 mean estimated transformed effect: 0.0038097649105030187


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


## 14. Qini Evaluation

In [15]:

from src.evaluation import uplift_metrics
for arm in ["Mens E-Mail","Womens E-Mail"]:
    mt,mc=train_tlearner(Xtr,train.segment.values,train.conversion.values,arm)
    pt,pc=predict_tlearner(mt,mc,Xte)
    metrics,qini=uplift_metrics(test.conversion.values,test.segment.values,pt-pc,arm)
    print(arm,metrics)


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Mens E-Mail {'auuc': 0.0019198953950943618, 'qini': 0.004128459154333266, 'uplift_at_10': 0.00642378603878312, 'uplift_at_20': 0.0001577906534180329, 'uplift_at_30': 0.0018572797341753627}


Womens E-Mail {'auuc': 0.0012460360371112247, 'qini': 0.0025983806095078864, 'uplift_at_10': 0.009274407579492324, 'uplift_at_20': 0.005336921650976537, 'uplift_at_30': 0.004748261627952865}


/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/pyvenv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## 15. AUUC and Uplift@10/20/30

In [16]:

comparison=pd.read_csv(ROOT/"artifacts/metrics/model_comparison.csv")
display(comparison.round(6))


,comparison,model,arm,auuc,qini,uplift_at_10,uplift_at_20,uplift_at_30
0,Mens E-Mail vs Control,Two-Model,Mens E-Mail,0.001920,0.004128,0.006424,0.000158,0.001857
1,Mens E-Mail vs Control,Class Transformation,Mens E-Mail,0.002054,0.004129,0.006279,0.004104,0.005486
2,Womens E-Mail vs Control,Two-Model,Womens E-Mail,0.001246,0.002598,0.009274,0.005337,0.004748
3,Womens E-Mail vs Control,Class Transformation,Womens E-Mail,0.001183,0.002414,0.003621,0.002580,0.004313
4,Any Email vs Control,Two-Model,Any Email,0.001514,0.002703,0.006843,0.004495,0.001434


## 16. Customer Ranking

In [17]:

pred=pd.read_csv(ROOT/"artifacts/predictions/customer_predictions.csv")
display(pred.sort_values("max_uplift",ascending=False)[["row_id","segment","uplift_mens","uplift_womens","recommended_action"]].head(20))


,row_id,segment,uplift_mens,uplift_womens,recommended_action
8264,8264,No E-Mail,0.610228,0.003254,Mens E-Mail
18437,18437,No E-Mail,0.610228,0.003254,Mens E-Mail
40625,40625,No E-Mail,0.610228,0.003254,Mens E-Mail
21820,21820,Mens E-Mail,0.610228,0.003254,Mens E-Mail
9394,9394,No E-Mail,0.610225,0.003380,Mens E-Mail
6792,6792,Womens E-Mail,0.610215,0.003369,Mens E-Mail
16364,16364,Mens E-Mail,0.516362,0.030838,Mens E-Mail
23435,23435,Mens E-Mail,0.492529,0.000558,Mens E-Mail
52951,52951,Womens E-Mail,0.040154,0.471620,Womens E-Mail
30063,30063,Womens E-Mail,0.000928,0.471620,Womens E-Mail


## 17. Personalized Treatment Policy

In [18]:

from src.policy import apply_policy
policy=apply_policy(pred)
display(policy["recommended_action"].value_counts())


recommended_action
Mens E-Mail      39486
Womens E-Mail    19057
No E-Mail         5457
Name: count, dtype: int64

## 18. Business Impact

In [19]:

curve=pd.read_csv(ROOT/"artifacts/metrics/targeting_curve.csv")
display(curve)


,targeting_depth,customers_targeted,mens_emails,womens_emails,no_email,expected_incremental_conversions,expected_incremental_revenue,average_uplift
0,0.05,800,483,317,15200,-6.031858,-304.351142,0.085831
1,0.10,1600,984,616,14400,17.962972,2981.000692,0.061408
2,0.20,3200,2036,1164,12800,17.948917,2708.367536,0.042598
3,0.30,4800,3143,1657,11200,8.911236,2297.245344,0.033558
4,0.40,6400,4262,2138,9600,8.897746,3098.774658,0.027916
5,0.50,8000,5314,2686,8000,41.879679,6050.920445,0.023945
6,1.00,14600,9838,4762,1400,26.807884,5376.537425,0.014832


The business-impact estimates use held-out inverse-propensity weighting. Because campaign costs are not supplied, incremental revenue is not converted into net profit.

## 19. Final Recommendation

In [20]:

best_model=comparison.loc[comparison.auuc.idxmax()]
best_depth=curve.loc[curve.expected_incremental_revenue.idxmax()]
print("Best model:",best_model["model"],best_model["comparison"])
print("Best revenue-oriented depth:",best_depth["targeting_depth"])
print("Estimated incremental conversions:",best_depth["expected_incremental_conversions"])
print("Estimated incremental revenue:",best_depth["expected_incremental_revenue"])


Best model: Class Transformation Mens E-Mail vs Control
Best revenue-oriented depth: 0.5
Estimated incremental conversions: 41.87967936787787
Estimated incremental revenue: 6050.920444945045


## 20. Limitations

Individual treatment effects are estimates rather than observed causal truths. Policy evaluation has sampling uncertainty. Spend is zero-inflated, campaign costs are unavailable, and production deployment can face distribution shift.

## 21. Conclusion

The project demonstrates the progression from randomized experimentation to causal effect estimation, heterogeneous uplift modeling, treatment personalization, and business policy evaluation.